In [1]:
########################################################
###################### Importierte Packete #############
########################################################

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rc                                                   # To use a LaTeX font
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.colors import rgb2hex

#import csv

import numpy as np

import pandas as pd

from IPython.display import display, HTML

import scipy
from scipy import stats
import scipy.constants as sc
from scipy import ndimage
import scipy.signal
from scipy.interpolate import interp1d

import statsmodels.api as sm

from mpl_toolkits import mplot3d                                            # Um 3D Plots zu machen

import os

##### Saving/packing of data
import pickle

##### To create colour gradients
from colour import Color

In [2]:
# importing the own functions from the scientific_utils package
from scientific_utils.general.plotting import Convert_cm_To_inch

In [3]:
#########################################################
##################### Start of the code ##################
#########################################################

In [4]:
########## Open the saved lists from the .pkl files ########

##### 
with open('../data/Batt15/processed/dataframes_of_charge_discharge_capacities_per_rate.pkl', 'rb') as f:
    dataframes_of_charge_discharge_capacities_per_rate = pickle.load(f)    

    
    
##### 
with open('../data/Batt15/processed/dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_rate.pkl', 'rb') as f:
    dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_rate = pickle.load(f) 

##### 
with open('../data/Batt15/processed/dataframes_of_potential_drop_between_CP_and_OCV_per_rate.pkl', 'rb') as f:
    dataframes_of_potential_drop_between_CP_and_OCV_per_rate = pickle.load(f) 

    
    
#### 
with open('../data/Batt15/processed/dataframe_of_dQdV_CP_charge_peaks.pkl', 'rb') as f:
    dataframe_of_dQdV_CP_charge_peaks = pickle.load(f) 
    
#### 
with open('../data/Batt15/processed/dataframe_of_dQdV_CP_discharge_peaks.pkl', 'rb') as f:
    dataframe_of_dQdV_CP_discharge_peaks = pickle.load(f) 
    

    
#### 
with open('../data/Batt15/processed/dataframe_of_dQdV_CP_charge_peaks_sorted.pkl', 'rb') as f:
    dataframe_of_dQdV_CP_charge_peaks_sorted = pickle.load(f) 
    
#### 
with open('../data/Batt15/processed/dataframe_of_dQdV_CP_discharge_peaks_sorted.pkl', 'rb') as f:
    dataframe_of_dQdV_CP_discharge_peaks_sorted = pickle.load(f)     



########## Create a list with more information about the data for plotting later #######
list_RateTest_labels = [0.10, 0.33, 1.00]


In [5]:
########## Define colours by hex code ##########
yellow = '#ffa200'
orange = '#f97625'
lightpink = '#e94a3a'
pink = '#ce1b49'
lightpurple = '#a90054'
purple = '#7e0059'
darkpurple = '#4c0057'
darkblue = '#0a004d'

########## Colours in a list to access them in a for loop ##########
colors = [yellow, orange, lightpink, pink, lightpurple, purple, darkpurple, darkblue]


########## Colour gradient with the colour package
#beginnig_color = Color("yellow")
#end_color = Color("blue")
#number_of_colorpoints = 4
#color_gradient = list(beginnig_color.range_to(end_color,number_of_colorpoints))
#print(color_gradient)


########## Cologradient with Matplotlib Colormaps
# Available colormaps: https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html#ibm
number_of_colorpoints = 10 # number of colour points taken from the colour gradient with equal distance
colormap = plt.get_cmap('coolwarm', number_of_colorpoints) # Loads a ready-made colormap from Matplotlib
array_of_RGBAcolors = colormap(range(number_of_colorpoints)) # Returns the colormap as RGBA values scaled from 0 to 1
list_of_Hexcolors = [ rgb2hex(array_of_RGBAcolors[i,:]) for i in range(array_of_RGBAcolors.shape[0]) ] # Converts the scaled RGBA values to hex code and puts them in a list


In [6]:
########## Font settings for the PGF file ##########
pgf_with_latex = {                      # setup matplotlib to use latex for output
    "pgf.texsystem": "xelatex",         # change this if using xetex or lautex
    "text.usetex": False,                # use LaTeX to write all text
    #"pgf.rcfonts": False,               # don't setup fonts from rc parameters
    "font.family": "sans-serif",           # standard sans-serif font of Matplotlib
    #"mathtext.fontset": "stixsans",
    "legend.title_fontsize" : 14,
    "legend.fontsize": 10,              # Make the legend/label fonts a little smaller
    "axes.labelsize": 12,               # LaTeX default is 10pt font.
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.figsize" : Convert_cm_To_inch(16, 8),
    "pgf.preamble": "\n".join([
        r"\usepackage[T1]{fontenc}",    # use utf8 fonts becasue your computer can handle it :)
        r"\usepackage{fontspec}",        # plots will be generated using this preamble
        r"\usepackage{fourier}",
        r"\usepackage{unicode-math}",
        r"\usepackage[basic,italic,symbolgreek,numbers]{mathastext}"
        ])
    }

plt.rcParams.update(pgf_with_latex)

##### one style for all plots: the shared publication style (scientific_utils.general.plotting) replaces the local settings above
from scientific_utils.general.plotting import pgf_with_latex
plt.rcParams.update(pgf_with_latex)

In [7]:
###################################### Definition of Function ########################################

In [8]:
########## To delete the file with the general data of all batteries and save it again
    
dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging = pd.DataFrame() # creating new empty dataframe
#dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.append(pd.Series(name=0))

#print(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)
    
##### General_Data_from_every_Cell
with open('../data/Batt15/processed/dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.pkl', 'wb') as f:
    pickle.dump(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging, f)

In [9]:
# Contacting the dQdV Peaks of charge and discharge steps in the datatframe with all the Rate Test data
dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging = pd.concat([dataframe_of_dQdV_CP_charge_peaks_sorted, dataframe_of_dQdV_CP_discharge_peaks_sorted], axis=1)

display(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)

,0.10 C 1. dQdV peak charge / mAh/V charge,0.10 C 1. Peak Savgol-Filter <Ewe>/V charge,0.10 C 2. dQdV peak charge / mAh/V charge,0.10 C 2. Peak Savgol-Filter <Ewe>/V charge,0.10 C 3. dQdV peak charge / mAh/V charge,0.10 C 3. Peak Savgol-Filter <Ewe>/V charge,0.10 C 4. dQdV peak charge / mAh/V charge,0.10 C 4. Peak Savgol-Filter <Ewe>/V charge,0.33 C 1. dQdV peak charge / mAh/V charge,0.33 C 1. Peak Savgol-Filter <Ewe>/V charge,...,0.10 C 4. dQdV peak charge / mAh/V discharge,0.10 C 4. Peak Savgol-Filter <Ewe>/V discharge,0.33 C 1. dQdV peak charge / mAh/V discharge,0.33 C 1. Peak Savgol-Filter <Ewe>/V discharge,0.33 C 2. dQdV peak charge / mAh/V discharge,0.33 C 2. Peak Savgol-Filter <Ewe>/V discharge,0.33 C 3. dQdV peak charge / mAh/V discharge,0.33 C 3. Peak Savgol-Filter <Ewe>/V discharge,1.0 C 1. dQdV peak charge / mAh/V discharge,1.0 C 1. Peak Savgol-Filter <Ewe>/V discharge
0,5.257923,3.037338,NaN,NaN,2.056541,3.392087,2.628995,3.590042,3.614977,3.171908,...,NaN,NaN,-1.699263,3.294326,-2.382295,3.047909,NaN,NaN,-1.663053,2.971375


In [10]:
########## Saving the total charge and discharge capaciteies in the dataframe ##########
### charging steps of the rate test
labels_charge_capacities = ['0.10 C Rate total charge capacity / mAh', '0.33 C Rate total charge capacity / mAh', '1.00 C Rate total charge capacity / mAh'] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_rate)):
    dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.loc[0, labels_charge_capacities[i]] = dataframes_of_charge_discharge_capacities_per_rate.iloc[i]['Charge Capacity / mAh'] # A new value is saved in the dataframe by index and column name. If the index and column already exist, the value is replaced, otherwise a new column is created

### discharging steps of the rate test
labels_discharge_capacities = ['0.10 C Rate total discharge capacity / mAh', '0.33 C Rate total discharge capacity / mAh', '1.00 C Rate total discharge capacity / mAh']
for i in range(0, len(dataframes_of_charge_discharge_capacities_per_rate)):
    dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.loc[0, labels_discharge_capacities[i]] = dataframes_of_charge_discharge_capacities_per_rate.iloc[i]['Discharge Capacity / mAh']

#print(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)

In [11]:
########## Saving the potential drop between CP step and OCV ##########
### charging steps of the rate test
labels_charge_potential_drop_CP_to_OCV = ['0.10 C Rate DeltaV CP charge to OCV / V', '0.33 C Rate DeltaV CP charge to OCV / V', '1.00 C Rate DeltaV CP charge to OCV / V'] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_potential_drop_between_CP_and_OCV_per_rate)):
    dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.loc[0, labels_charge_potential_drop_CP_to_OCV[i]] = dataframes_of_potential_drop_between_CP_and_OCV_per_rate.iloc[i]['DeltaV CP charge to OCV'] # A new value is saved in the dataframe by index and column name. If the index and column already exist, the value is replaced, otherwise a new column is created

### discharging steps of the rate test
labels_discharge_potential_drop_CP_to_OCV = ['0.10 C Rate DeltaV CP discharge to OCV / V', '0.33 C Rate DeltaV CP discharge to OCV / V', '1.00 C Rate DeltaV CP discharge to OCV / V']
for i in range(0, len(dataframes_of_potential_drop_between_CP_and_OCV_per_rate)):
    dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.loc[0, labels_discharge_potential_drop_CP_to_OCV[i]] = dataframes_of_potential_drop_between_CP_and_OCV_per_rate.iloc[i]['DeltaV CP discharge to OCV']

#print(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)

In [12]:
########## Saving the potential difference between the beginning of the ocv step and after trhee hours##########
### OCV after charging steps of the rate test
labels_charge_potential_drop_OCV = ['0.10 C Rate charge; DeltaV OCV after three hours / V', '0.33 C Rate charge; DeltaV OCV after three hours / V', '1.00 C Rate charge; DeltaV OCV after three hours / V'] # List of labels used as column names of the dataframe
for i in range(0, len(dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_rate)):
    dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.loc[0, labels_charge_potential_drop_OCV[i]] = dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_rate.iloc[i]['Potential Difference OCV charge at end'] # A new value is saved in the dataframe by index and column name. If the index and column already exist, the value is replaced, otherwise a new column is created

### OCV after discharging steps of the rate test
labels_discharge_potential_drop_OCV = ['0.10 C Rate discharge; DeltaV OCV after three hours / V', '0.33 C Rate discharge; DeltaV OCV after three hours / V', '1.00 C Rate discharge; DeltaV OCV after three hours / V']
for i in range(0, len(dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_rate)):
    dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.loc[0, labels_discharge_potential_drop_OCV[i]] = dataframes_of_potential_drop_beginning_OCV_and_end_OCV_per_rate.iloc[i]['Potential Difference OCV discharge at end']

#print(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)

In [13]:
display(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging)

,0.10 C 1. dQdV peak charge / mAh/V charge,0.10 C 1. Peak Savgol-Filter <Ewe>/V charge,0.10 C 2. dQdV peak charge / mAh/V charge,0.10 C 2. Peak Savgol-Filter <Ewe>/V charge,0.10 C 3. dQdV peak charge / mAh/V charge,0.10 C 3. Peak Savgol-Filter <Ewe>/V charge,0.10 C 4. dQdV peak charge / mAh/V charge,0.10 C 4. Peak Savgol-Filter <Ewe>/V charge,0.33 C 1. dQdV peak charge / mAh/V charge,0.33 C 1. Peak Savgol-Filter <Ewe>/V charge,...,1.00 C Rate DeltaV CP charge to OCV / V,0.10 C Rate DeltaV CP discharge to OCV / V,0.33 C Rate DeltaV CP discharge to OCV / V,1.00 C Rate DeltaV CP discharge to OCV / V,0.10 C Rate charge; DeltaV OCV after three hours / V,0.33 C Rate charge; DeltaV OCV after three hours / V,1.00 C Rate charge; DeltaV OCV after three hours / V,0.10 C Rate discharge; DeltaV OCV after three hours / V,0.33 C Rate discharge; DeltaV OCV after three hours / V,1.00 C Rate discharge; DeltaV OCV after three hours / V
0,5.257923,3.037338,NaN,NaN,2.056541,3.392087,2.628995,3.590042,3.614977,3.171908,...,0.189427,-0.077234,-0.254471,-0.578612,0.065669,0.079638,0.160593,-0.982299,-0.855438,-0.585659


In [14]:
########## Save the data ##########
with open('../data/Batt15/processed/dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging.pkl', 'wb') as f:
    pickle.dump(dataframe_with_all_extracted_data_from_Rate_Test_OCV_Charging_Discharging, f)